In [13]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Credit Card Fraud Dataset Exploration

This notebook explores the Kaggle Credit Card Fraud Detection dataset containing 284,807 real credit card transactions made by European cardholders over two days in September 2013. Out of those ~285k transactions, only 492 are fraudulent (0.17%).

## Dataset Overview
- **Real data** from real transactions (anonymized for privacy)
- **Extreme class imbalance** - fraud is rare (0.17%)
- **Standard benchmark** for fraud detection projects
- **Features**: V1-V28 (PCA-anonymized), Time, Amount, Class (label)

In [14]:
df = pd.read_csv("creditcard.csv")
print(f"Dataset loaded successfully!")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Dataset loaded successfully!
Shape: (284807, 31)
Columns: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']


## Column Descriptions

| Column | What it is | Range |
|---|---|---|
| `Time` | Seconds elapsed since the first transaction (relative offset, ~2 days) | 0 to ~172,792 |
| `V1` through `V28` | PCA-anonymized transaction features (location, merchant category, patterns, etc.) | Roughly -5 to +5 |
| `Amount` | Transaction amount in euros | 0 to 25,691.16 |
| `Class` | Label: 0 = legitimate, 1 = fraud | 0 or 1 |

**Important**: V1-V28 are already normalized by PCA (mean≈0, std≈1). `Time` and `Amount` need normalization.

In [15]:
missing = df.isnull().sum().sum()
print(f"Total missing values: {missing}")
print("✓ Clean dataset - no missing values!")

Total missing values: 0
✓ Clean dataset - no missing values!


## Class Distribution: The Imbalance Problem

Class imbalance is the central challenge here. Fraud is rare (0.17%), so a naive "always predict legitimate" model would have 99.83% accuracy while catching zero fraud. We must use appropriate metrics (Precision, Recall, F1) instead of accuracy.

In [16]:
counts = df['Class'].value_counts()
print("CLASS DISTRIBUTION")
print("=" * 50)
print(f"Legitimate (0): {counts[0]:,}  ({counts[0]/len(df)*100:.2f}%)")
print(f"Fraud (1):      {counts[1]:,}      ({counts[1]/len(df)*100:.2f}%)")
print(f"Ratio:          {counts[0]//counts[1]}:1")

CLASS DISTRIBUTION
Legitimate (0): 284,315  (99.83%)
Fraud (1):      492      (0.17%)
Ratio:          577:1


In [17]:
print("AMOUNT STATISTICS (before normalization)")
print(f"Mean:   ${df['Amount'].mean():.2f}")
print(f"Median: ${df['Amount'].median():.2f}")
print(f"Std:    ${df['Amount'].std():.2f}")
print(f"Min:    ${df['Amount'].min():.2f}")
print(f"Max:    ${df['Amount'].max():.2f}")

print("\n-- Amount by class --")
for cls in [0, 1]:
    subset = df[df['Class'] == cls]['Amount']
    label = "Legitimate" if cls == 0 else "Fraud"
    print(f"{label:12}: mean=${subset.mean():>8.2f}, median=${subset.median():>8.2f}, max=${subset.max():>10.2f}")

AMOUNT STATISTICS (before normalization)
Mean:   $88.35
Median: $22.00
Std:    $250.12
Min:    $0.00
Max:    $25691.16

-- Amount by class --
Legitimate  : mean=$   88.29, median=$   22.00, max=$  25691.16
Fraud       : mean=$  122.21, median=$    9.25, max=$   2125.87


In [18]:
print("TIME STATISTICS (before normalization)")
print(f"Mean:  {df['Time'].mean():.0f} seconds ({df['Time'].mean()/3600:.1f} hours)")
print(f"Min:   {df['Time'].min():.0f} seconds")
print(f"Max:   {df['Time'].max():.0f} seconds ({df['Time'].max()/3600:.1f} hours)")

TIME STATISTICS (before normalization)
Mean:  94814 seconds (26.3 hours)
Min:   0 seconds
Max:   172792 seconds (48.0 hours)


In [19]:
print("V-FEATURE RANGES (already PCA-scaled)")
v_cols = [c for c in df.columns if c.startswith('V')]
print(f"Checking first 5 of {len(v_cols)} V-features:\n")
for col in v_cols[:5]:
    print(f"{col:>4}: mean={df[col].mean():>8.4f}  std={df[col].std():>6.4f}  "
          f"min={df[col].min():>8.2f}  max={df[col].max():>8.2f}")

V-FEATURE RANGES (already PCA-scaled)
Checking first 5 of 28 V-features:

  V1: mean=  0.0000  std=1.9587  min=  -56.41  max=    2.45
  V2: mean=  0.0000  std=1.6513  min=  -72.72  max=   22.06
  V3: mean= -0.0000  std=1.5163  min=  -48.33  max=    9.38
  V4: mean=  0.0000  std=1.4159  min=   -5.68  max=   16.88
  V5: mean=  0.0000  std=1.3802  min= -113.74  max=   34.80


In [20]:
print("TOP FEATURES CORRELATED WITH FRAUD")
corr = df.corr()['Class'].drop('Class').sort_values()
print("\nMost negatively correlated (high value = LESS likely fraud):")
for feat, val in corr.head(5).items():
    print(f"  {feat:>8}: {val:.4f}")
print("\nMost positively correlated (high value = MORE likely fraud):")
for feat, val in corr.tail(5).items():
    print(f"  {feat:>8}: {val:.4f}")

TOP FEATURES CORRELATED WITH FRAUD

Most negatively correlated (high value = LESS likely fraud):
       V17: -0.3265
       V14: -0.3025
       V12: -0.2606
       V10: -0.2169
       V16: -0.1965

Most positively correlated (high value = MORE likely fraud):
       V19: 0.0348
       V21: 0.0404
        V2: 0.0913
        V4: 0.1334
       V11: 0.1549


## Normalization: Why We Need It

Many ML algorithms (including Isolation Forest) make decisions based on **distance**. When features have different scales:
- V-features: -5 to +5
- Amount: 0 to 25,691  
- Time: 0 to 172,792

The large-scale features (Amount, Time) dominate the math, drowning out smaller features.

**Solution**: StandardScaler transforms each feature to have mean=0 and std=1:
$$\text{normalized\_value} = \frac{\text{original\_value} - \text{mean}}{\text{std}}$$

This puts all features on the same scale.

In [21]:
print("NORMALIZATION")
print(f"\nBEFORE normalization:")
print(f"  Amount — mean: {df['Amount'].mean():.2f}, std: {df['Amount'].std():.2f}")
print(f"  Time   — mean: {df['Time'].mean():.2f}, std: {df['Time'].std():.2f}")

# Create scalers and fit/transform
scaler_amount = StandardScaler()
scaler_time = StandardScaler()

df['Amount_scaled'] = scaler_amount.fit_transform(df['Amount'].values.reshape(-1, 1))
df['Time_scaled'] = scaler_time.fit_transform(df['Time'].values.reshape(-1, 1))

# Drop original unscaled columns
df = df.drop(columns=['Amount', 'Time'])

print(f"\nAFTER normalization:")
print(f"  Amount_scaled — mean: {df['Amount_scaled'].mean():.6f}, std: {df['Amount_scaled'].std():.6f}")
print(f"  Time_scaled   — mean: {df['Time_scaled'].mean():.6f}, std: {df['Time_scaled'].std():.6f}")

NORMALIZATION

BEFORE normalization:
  Amount — mean: 88.35, std: 250.12
  Time   — mean: 94813.86, std: 47488.15

AFTER normalization:
  Amount_scaled — mean: -0.000000, std: 1.000002
  Time_scaled   — mean: -0.000000, std: 1.000002


In [22]:
# Final dataset state
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst 3 rows:")
print(df.head(3).to_string())
print(f"\nMeans of all features (should all be near 0):")
print(df.drop(columns=['Class']).mean().round(6))
print("\n✓ Dataset exploration and normalization complete!")

Shape: (284807, 31)
Columns: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Class', 'Amount_scaled', 'Time_scaled']

First 3 rows:
         V1        V2        V3        V4        V5        V6        V7        V8        V9       V10       V11       V12       V13       V14       V15       V16       V17       V18       V19       V20       V21       V22       V23       V24       V25       V26       V27       V28  Class  Amount_scaled  Time_scaled
0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599  0.098698  0.363787  0.090794 -0.551600 -0.617801 -0.991390 -0.311169  1.468177 -0.470401  0.207971  0.025791  0.403993  0.251412 -0.018307  0.277838 -0.110474  0.066928  0.128539 -0.189115  0.133558 -0.021053      0       0.244964    -1.996583
1  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803  0.085102 -0.255425 -